# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.
2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 
3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?
4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?
6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?

Be mindful that you will be using your experiments to write up your final blog, so make sure your code is clean and neat!

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)

# Import Paws again
We use paws again for thsi project

## Install project requirements

Run the next cell once. %pip installs packages into the active notebook kernel. Restart the kernel when installation finishes, then run the notebook from the top.

In [1]:
%pip install -r ../requirements.txt

  Using cached scipy-1.13.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached gensim-4.4.0-cp311-cp311-win_amd64.whl.metadata (8.6 kB)
  Using cached wordfreq-3.1.1-py3-none-any.whl.metadata (27 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
  Using cached wrapt-2.5.2-cp311-cp311-win_amd64.whl.metadata (7.7 kB)
  Using cached ftfy-6.3.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached langcodes-3.5.1-py3-none-any.whl.metadata (30 kB)
  Using cached locate-1.1.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached msgpack-1.2.4-cp311-cp311-win_amd64.whl.metadata (8.5 kB)
  Using cached babel-2.18.0-py3-none-any.whl.metadata (2.2 kB)
Using cached scipy-1.13.1-cp311-cp311-win_amd64.whl (46.2 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
   ---------------------------------------- 0.0/3.1 MB ? eta 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location

In [2]:
import torch
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import part1
from datasets import load_dataset

dataset = load_dataset("google-research-datasets/paws", "labeled_final")
paws_outcome_space = part1.clslib.NamedClasses(['not paraphrase', 'paraphrase'])


# Build the same sentence-pair folds used by the earlier experiments.
pairs = list(zip(
    dataset["train"]["sentence1"],
    dataset["train"]["sentence2"],
    dataset["train"]["label"],
))
num_folds = 5
my_cross_validation_sets = [
    part1.experiment.cross_validation_split(pairs, k=num_folds, seed=0)
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# importing embedders

In [3]:
import sys
import urllib.request
import zipfile
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import part2

GLOVE_URL = "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip"


def load_glove(dims):
    """
    Load the GloVe 6B vectors of size `dims` (50, 100, 200 or 300).
    Downloads the zip (~822 MB) once and only extracts the file we need.
    """
    glove_zip = Path("glove.6B.zip")
    glove_txt = Path(f"glove.6B.{dims}d.txt")

    if not glove_txt.exists():
        if not glove_zip.exists() or not zipfile.is_zipfile(glove_zip):
            print("Downloading GloVe 6B (~822 MB) from the Stanford NLP mirror on Hugging Face...")
            glove_zip.unlink(missing_ok=True)
            urllib.request.urlretrieve(GLOVE_URL, glove_zip)
        if not zipfile.is_zipfile(glove_zip):
            raise RuntimeError("The downloaded GloVe file is not a valid ZIP archive.")
        with zipfile.ZipFile(glove_zip) as z:
            z.extract(glove_txt.name)

    embeddings = part2.clslib.PretrainedEmbeddings(str(glove_txt), dims)
    print(f"GloVe {dims}d: vocab size {len(embeddings.vocab):,}, dims {embeddings.num_dims}")
    return embeddings


GLOVE_DIMS = 100  # the 300d ablation is in its own section further down
glove = load_glove(GLOVE_DIMS)

c:\Users\leonardo\miniconda3\envs\optimus\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


vocab size: 400,002, dims: 100


In [4]:
import numpy as np


class RandomEmbeddings:
    """Same vocabulary and size as `reference_embeddings`, but random vectors: the no-pretraining baseline."""

    def __init__(self, reference_embeddings, seed=0):
        self.vocab = reference_embeddings.vocab
        self.num_dims = reference_embeddings.num_dims

        rng = np.random.default_rng(seed)
        self.vectors = rng.uniform(
            low=-0.05,
            high=0.05,
            size=(len(self.vocab), self.num_dims),
        ).astype(np.float32)

## Saving results

Every run writes its fold accuracies to `results/fold_scores.json`, so a kernel restart does not lose them and the comparison section at the end can run on its own. The first six entries were copied from the printed outputs of our first run (rounded to 4 decimals). Re-running a cell overwrites its entry.

For the error analysis we also save, for three systems, whether each pair was classified correctly by the fold model that did not train on it (`results/correct_<name>.npy`).

In [ ]:
import json

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
FOLD_SCORES_FILE = RESULTS_DIR / "fold_scores.json"


def load_fold_scores():
    """Return {run name: [accuracy of fold 0, ..., accuracy of fold 4]} for every run saved so far."""
    if not FOLD_SCORES_FILE.exists():
        return {}
    return json.loads(FOLD_SCORES_FILE.read_text())


def save_fold_scores(name, results):
    """Add the per-fold accuracies of one run (the list returned by our training functions) to the results file."""
    scores = load_fold_scores()
    scores[name] = [result["accuracy"] for result in sorted(results, key=lambda result: result["fold"])]
    FOLD_SCORES_FILE.write_text(json.dumps(scores, indent=2))
    print(f"saved '{name}': mean accuracy {np.mean(scores[name]):.4f}")


def save_correctness(name, correct):
    """Save one 0/1 flag per PAWS training pair (1 = classified correctly when it was in the validation fold)."""
    np.save(RESULTS_DIR / f"correct_{name}.npy", correct)
    print(f"saved correctness of '{name}': {correct.mean():.4f} of pairs correct")


def load_correctness(name):
    return np.load(RESULTS_DIR / f"correct_{name}.npy")

# 1. Is GloVe a good choice for PAWS?

We use **GloVe 6B** (Pennington et al., 2014): 400k lowercased words, trained on Wikipedia 2014 and Gigaword 5. PAWS (labeled_final) is built from Wikipedia sentences, so the domain matches the training data of the embeddings.

Tokenization: GloVe 6B is lowercased and tokenized with the Stanford tokenizer, and `PretrainedEmbeddings` lowercases every word it loads. Our `BasicTokenizer` lowercases too, removes punctuation and splits with NLTK, so the tokens we look up are the same kind of tokens GloVe has vectors for. We use exactly the tokenizer settings of the training functions below to measure how many of our words GloVe knows.

In [ ]:
from collections import Counter
from tqdm.auto import tqdm

# Same settings as the tokenizer inside our training functions
analysis_tokenizer = part2.clslib.BasicTokenizer(
    lowercase=True,
    remove_punct=True,
    remove_stopwords=False,
)

s1_all = list(dataset["train"]["sentence1"])
s2_all = list(dataset["train"]["sentence2"])
y_all = np.array(dataset["train"]["label"])
tokens1 = [analysis_tokenizer.tokens(sentence) for sentence in tqdm(s1_all, desc="tokenizing sentence 1")]
tokens2 = [analysis_tokenizer.tokens(sentence) for sentence in tqdm(s2_all, desc="tokenizing sentence 2")]

token_counts = Counter(token for sentence in tokens1 + tokens2 for token in sentence)
unknown_counts = Counter({token: count for token, count in token_counts.items() if token not in glove.vocab})
num_tokens = sum(token_counts.values())

print(f"word types in PAWS train: {len(token_counts):,}, tokens: {num_tokens:,}")
print(f"types not in GloVe:  {len(unknown_counts):,} ({len(unknown_counts) / len(token_counts):.1%})")
print(f"tokens not in GloVe: {sum(unknown_counts.values()):,} ({sum(unknown_counts.values()) / num_tokens:.2%})")


def unknown_rate(tokens):
    """Fraction of tokens that GloVe has no vector for (they all get the <unk> vector)."""
    return sum(token not in glove.vocab for token in tokens) / max(len(tokens), 1)


pair_unknown_rate = np.array([unknown_rate(t1 + t2) for t1, t2 in zip(tokens1, tokens2)])
print(f"pairs with at least one unknown word: {(pair_unknown_rate > 0).mean():.1%}")
print("\nmost frequent unknown words:")
print(unknown_counts.most_common(25))

print("\nexample (unknown words in [brackets]):")
for tokens in (tokens1[0], tokens2[0]):
    print("  " + " ".join(token if token in glove.vocab else f"[{token}]" for token in tokens))

# 2. What does GloVe capture, and what does it miss?

PAWS is built by **swapping words** and **back-translation**. A non-paraphrase often uses the same words as its partner in a different order ("Flights from New York to Florida" vs "Flights from Florida to New York"). Two things matter for this task:

1. **Similar is not the same.** GloVe learns from co-occurrence, so words that appear in the same contexts get similar vectors, also when they have opposite or swapped roles (before/after, north/south, husband/wife). Below we compare such meaning-flipping pairs with synonym pairs.
2. **Mean pooling throws away word order.** If both sentences contain the same words, their mean-pooled vectors are identical, whatever the order. We measure how often that happens in PAWS.

In [ ]:
import matplotlib.pyplot as plt

TRAIN_COLOUR, VALIDATION_COLOUR = "#2E5FA3", "#B84A12"  # same colours as in Week 1

unit_vectors = glove.vectors / (np.linalg.norm(glove.vectors, axis=1, keepdims=True) + 1e-8)


def cosine(word_a, word_b):
    return float(unit_vectors[glove.index(word_a)] @ unit_vectors[glove.index(word_b)])


def nearest_neighbours(word, k=8):
    """The k words with the highest cosine similarity to `word` (skipping <pad>, <unk> and the word itself)."""
    similarity = unit_vectors @ unit_vectors[glove.index(word)]
    similarity[[0, 1, glove.index(word)]] = -np.inf
    return [glove.word(i) for i in np.argsort(-similarity)[:k]]


print("Nearest neighbours")
for word in ["before", "north", "husband", "won", "older", "film"]:
    print(f"  {word:8s} -> {', '.join(nearest_neighbours(word))}")

meaning_flips = [("before", "after"), ("north", "south"), ("east", "west"), ("older", "younger"),
                 ("first", "last"), ("husband", "wife"), ("father", "son"), ("won", "lost"),
                 ("increase", "decrease"), ("left", "right")]
synonyms = [("start", "begin"), ("big", "large"), ("city", "town"), ("film", "movie"),
            ("quickly", "rapidly"), ("buy", "purchase"), ("rich", "wealthy")]

print("\nCosine similarity")
for title, word_pairs in (("meaning flips", meaning_flips), ("synonyms", synonyms)):
    similarities = [cosine(a, b) for a, b in word_pairs]
    print(f"  {title}: mean {np.mean(similarities):.2f}")
    print("    " + ", ".join(f"{a}/{b} {s:.2f}" for (a, b), s in zip(word_pairs, similarities)))


# Word order: mean-pooled sentence vectors of both sentences in a pair
def mean_pooled(tokens):
    if not tokens:
        return np.zeros(glove.num_dims, dtype=np.float32)
    return glove.vectors[[glove.index(token) for token in tokens]].mean(axis=0)


pooled1 = np.stack([mean_pooled(tokens) for tokens in tokens1])
pooled2 = np.stack([mean_pooled(tokens) for tokens in tokens2])
pooled_cosine = (pooled1 * pooled2).sum(1) / (np.linalg.norm(pooled1, axis=1) * np.linalg.norm(pooled2, axis=1) + 1e-8)
same_bag = np.array([Counter(t1) == Counter(t2) for t1, t2 in zip(tokens1, tokens2)])
jaccard = np.array([len(set(t1) & set(t2)) / max(len(set(t1) | set(t2)), 1) for t1, t2 in zip(tokens1, tokens2)])

print("\nMean pooling and word order")
print(f"  pairs whose sentences contain exactly the same words: {same_bag.mean():.1%}")
print(f"  of those, paraphrases: {y_all[same_bag].mean():.1%}  (mean pooling gives both sentences the SAME vector)")
for label, name in enumerate(["not paraphrase", "paraphrase"]):
    mask = y_all == label
    print(f"  {name}: mean cosine of pooled vectors {pooled_cosine[mask].mean():.3f}, identical word bag {same_bag[mask].mean():.1%}")

low = np.floor(np.percentile(pooled_cosine, 2) * 1000) / 1000  # almost all pairs are close to 1, so zoom in
bins = np.linspace(low, 1.0, 21)
centres = (bins[:-1] + bins[1:]) / 2
fig, ax = plt.subplots(figsize=(8, 3.2))
for label, name, colour in ((0, "Not paraphrase", VALIDATION_COLOUR), (1, "Paraphrase", TRAIN_COLOUR)):
    counts, _ = np.histogram(np.clip(pooled_cosine[y_all == label], low, 1.0), bins=bins)
    ax.plot(centres, 100 * counts / (y_all == label).sum(), color=colour, linewidth=2, label=name)
ax.set_xlabel(f"cosine of the mean-pooled GloVe vectors of the two sentences (values below {low} in the first bin)")
ax.set_ylabel("% of pairs with that label")
ax.set_title("Mean-pooled GloVe cannot tell the labels apart", loc="left", fontsize=11)
ax.grid(axis="y", color="#E5E7EB")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

# mlp
MEan pooling


In [5]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


class DenseDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.as_tensor(features, dtype=torch.float32)
        self.labels = torch.as_tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


class SentencePairMLP(nn.Module):
    """MLP classifier for concatenated GloVe sentence representations."""

    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
            nn.LogSoftmax(dim=-1),
        )

    def forward(self, features):
        return self.network(features)


def glove_sentence(text, tokenizer, glove_model):
    """Mean-pool the GloVe vectors for the tokens in one sentence."""
    token_ids = tokenizer.ids(text)
    if not token_ids:
        return np.zeros(glove_model.num_dims, dtype=np.float32)
    vectors = glove_model.vectors[token_ids]
    return vectors.mean(axis=0).astype(np.float32)


def sentence_pair_features(examples, tokenizer, glove_model):
    """Create [sentence1 embedding; sentence2 embedding] features."""
    return np.stack(
        [
            np.concatenate(
                [
                    glove_sentence(sentence1, tokenizer, glove_model),
                    glove_sentence(sentence2, tokenizer, glove_model),
                ]
            )
            for sentence1, sentence2, _ in examples
        ]
    ).astype(np.float32)


def run_epoch(model, loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = total_correct = total_examples = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            logprobs = model(features)
            loss = criterion(logprobs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            total_correct += (logprobs.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


def cross_validation_train_glove_mlp(
    embedding_model,
    my_cross_validation_sets,
    num_folds,
    num_classes=2,
    hidden_dim=128,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_epochs=20,
    device=None,
    seed=0,
    name="GloVe",
):
    """Train a sentence-pair MLP using mean-pooled GloVe embeddings."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device(device)
    torch.manual_seed(seed)  # same initialisation and batch order every time we run this

    tokenizer = part2.clslib.BasicTokenizer(
        lowercase=True,
        remove_punct=True,
        remove_stopwords=False,
        vocab=embedding_model.vocab,
    )
    results, models = [], {}

    for fold_index, split in my_cross_validation_sets[0].items():
        training_set = split["training"]
        validation_set = split["validation"]
        X_train = sentence_pair_features(training_set, tokenizer, embedding_model)
        X_validation = sentence_pair_features(validation_set, tokenizer, embedding_model)
        y_train = [int(example[2]) for example in training_set]
        y_validation = [int(example[2]) for example in validation_set]

        train_loader = DataLoader(
            DenseDataset(X_train, y_train), batch_size=batch_size, shuffle=True
        )
        validation_loader = DataLoader(
            DenseDataset(X_validation, y_validation), batch_size=batch_size
        )
        model = SentencePairMLP(
            input_dim=2 * embedding_model.num_dims,
            hidden_dim=hidden_dim,
            num_classes=num_classes,
            dropout=dropout,
        ).to(device)
        criterion = nn.NLLLoss()
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=learning_rate, weight_decay=weight_decay
        )

        best_state = None
        best_metrics = None
        best_epoch = 0
        for epoch in tqdm(range(num_epochs), desc=f"{name} fold {fold_index + 1}/{num_folds}"):
            train_metrics = run_epoch(model, train_loader, criterion, device, optimizer)
            validation_metrics = run_epoch(model, validation_loader, criterion, device)
            if best_metrics is None or validation_metrics["loss"] < best_metrics["loss"]:
                best_metrics = validation_metrics
                best_epoch = epoch
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }

        model.load_state_dict(best_state)
        models[fold_index] = model
        results.append({
            "fold": fold_index,
            "accuracy": best_metrics["accuracy"],
            "best_epoch": best_epoch,
        })
        print(
            f"Fold {fold_index + 1}/{num_folds}: "
            f"{best_metrics['accuracy']:.4f} "
            f"(best epoch {best_epoch + 1})"
        )

    print(f"Mean accuracy: {np.mean([result['accuracy'] for result in results]):.4f}")
    return results, models

In [ ]:
def mlp_validation_correctness(models, my_cross_validation_sets, embedding_model, batch_size=256):
    """
    For every PAWS training pair: 1 if the fold model that did NOT train on it predicts its label, else 0.
    Every pair sits in exactly one validation fold, so we get one out-of-fold prediction per pair,
    in the same order as dataset["train"].
    """
    tokenizer = part2.clslib.BasicTokenizer(
        lowercase=True,
        remove_punct=True,
        remove_stopwords=False,
        vocab=embedding_model.vocab,
    )
    correct = np.zeros(sum(len(split["validation"]) for split in my_cross_validation_sets[0].values()), dtype=int)

    for fold_index, split in my_cross_validation_sets[0].items():
        model = models[fold_index].eval()
        model_device = next(model.parameters()).device
        features = torch.as_tensor(sentence_pair_features(split["validation"], tokenizer, embedding_model))
        labels = np.array([int(label) for _, _, label in split["validation"]])

        with torch.no_grad():
            predictions = torch.cat([
                model(features[i:i + batch_size].to(model_device)).argmax(dim=1).cpu()
                for i in range(0, len(features), batch_size)
            ]).numpy()

        validation_ids = np.asarray(split["validation_ids"], dtype=int)
        correct[validation_ids] = (predictions == labels).astype(int)

    return correct

Glove
Mean pooling


In [6]:
# GloVe baseline: the fold examples already contain sentence pairs and labels.
glove_results, glove_models = cross_validation_train_glove_mlp(
    embedding_model=glove,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    num_classes=len(paws_outcome_space),
    device=device,
    name="MLP GloVe",
)
save_fold_scores("mlp-glove100", glove_results)
save_correctness("mlp-glove100", mlp_validation_correctness(glove_models, my_cross_validation_sets, glove))

GloVe fold 1/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 1/5: 0.5799 (best epoch 14)


GloVe fold 2/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 2/5: 0.5739 (best epoch 9)


GloVe fold 3/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 3/5: 0.5698 (best epoch 8)


GloVe fold 4/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 4/5: 0.5724 (best epoch 8)


GloVe fold 5/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 5/5: 0.5831 (best epoch 12)
Mean accuracy: 0.5758


Random mean pooling

In [7]:
random_embedding_model = RandomEmbeddings(glove)

random_results, random_models = cross_validation_train_glove_mlp(
    embedding_model=random_embedding_model,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    num_classes=len(paws_outcome_space),
    device=device,
    name="MLP random",
)
save_fold_scores("mlp-random", random_results)

GloVe fold 1/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 1/5: 0.5604 (best epoch 4)


GloVe fold 2/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 2/5: 0.5644 (best epoch 8)


GloVe fold 3/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 3/5: 0.5617 (best epoch 12)


GloVe fold 4/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 4/5: 0.5623 (best epoch 13)


GloVe fold 5/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 5/5: 0.5679 (best epoch 9)
Mean accuracy: 0.5633


# LSTM pooling

In [8]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset

class SentencePairSequenceDataset(Dataset):
    def __init__(self, examples, tokenizer):
        self.examples = examples
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        sentence1, sentence2, label = self.examples[index]
        ids1 = self.tokenizer.ids(sentence1) or [0]
        ids2 = self.tokenizer.ids(sentence2) or [0]

        return (
            torch.tensor(ids1, dtype=torch.long),
            torch.tensor(ids2, dtype=torch.long),
            int(label),
        )


def lstm_collate(batch):
    sequences1, sequences2, labels = zip(*batch)

    lengths1 = torch.tensor([len(x) for x in sequences1], dtype=torch.long)
    lengths2 = torch.tensor([len(x) for x in sequences2], dtype=torch.long)

    sequences1 = pad_sequence(sequences1, batch_first=True, padding_value=0)
    sequences2 = pad_sequence(sequences2, batch_first=True, padding_value=0)
    labels = torch.tensor(labels, dtype=torch.long)

    return sequences1, lengths1, sequences2, lengths2, labels


class SentencePairLSTMMLP(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        hidden_dim=128,
        num_classes=2,
        dropout=0.1,
        bidirectional=True,
        freeze_embeddings=False,
    ):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=freeze_embeddings,
            padding_idx=0,
        )

        self.lstm = nn.LSTM(
            input_size=embedding_matrix.shape[1],
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bidirectional,
        )

        output_dim = hidden_dim * (2 if bidirectional else 1)

        self.classifier = nn.Sequential(
            nn.Linear(output_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
            nn.LogSoftmax(dim=-1),
        )

    def encode(self, sequences, lengths):
        embedded = self.embedding(sequences)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, (hidden, _) = self.lstm(packed)

        if self.lstm.bidirectional:
            representation = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            representation = hidden[-1]

        return representation

    def forward(self, sequences1, lengths1, sequences2, lengths2):
        representation1 = self.encode(sequences1, lengths1)
        representation2 = self.encode(sequences2, lengths2)

        features = torch.cat((representation1, representation2), dim=1)
        return self.classifier(features)


def run_lstm_epoch(model, loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for sequences1, lengths1, sequences2, lengths2, labels in loader:
            sequences1 = sequences1.to(device)
            lengths1 = lengths1.to(device)
            sequences2 = sequences2.to(device)
            lengths2 = lengths2.to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad()

            logprobs = model(
                sequences1, lengths1,
                sequences2, lengths2,
            )
            loss = criterion(logprobs, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total_correct += (logprobs.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


def cross_validation_train_lstm(
    embedding_model,
    my_cross_validation_sets,
    num_folds,
    hidden_dim=128,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_epochs=15,
    freeze_embeddings=False,
    device=None,
    seed=0,
    name="LSTM",
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed)  # same initialisation and batch order every time we run this

    tokenizer = part2.clslib.BasicTokenizer(
        lowercase=True,
        remove_punct=True,
        remove_stopwords=False,
        vocab=embedding_model.vocab,
    )

    results = []
    models = {}

    for fold_index, split in my_cross_validation_sets[0].items():
        training_set = split["training"]
        validation_set = split["validation"]

        train_loader = DataLoader(
            SentencePairSequenceDataset(training_set, tokenizer),
            batch_size=batch_size,
            shuffle=True,
            collate_fn=lstm_collate,
        )
        validation_loader = DataLoader(
            SentencePairSequenceDataset(validation_set, tokenizer),
            batch_size=batch_size,
            collate_fn=lstm_collate,
        )

        model = SentencePairLSTMMLP(
            embedding_matrix=embedding_model.vectors,
            hidden_dim=hidden_dim,
            num_classes=len(paws_outcome_space),
            dropout=dropout,
            freeze_embeddings=freeze_embeddings,
        ).to(device)

        criterion = nn.NLLLoss()
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )

        best_state = None
        best_metrics = None
        best_epoch = 0

        for epoch in tqdm(
            range(num_epochs),
            desc=f"{name} fold {fold_index + 1}/{num_folds}",
        ):
            run_lstm_epoch(
                model, train_loader, criterion, device, optimizer
            )
            validation_metrics = run_lstm_epoch(
                model, validation_loader, criterion, device
            )

            if best_metrics is None or validation_metrics["loss"] < best_metrics["loss"]:
                best_metrics = validation_metrics
                best_epoch = epoch
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }

        model.load_state_dict(best_state)
        models[fold_index] = model

        results.append({
            "fold": fold_index,
            "accuracy": best_metrics["accuracy"],
            "best_epoch": best_epoch,
        })

        print(
            f"Fold {fold_index + 1}/{num_folds}: "
            f"{best_metrics['accuracy']:.4f} "
            f"(best epoch {best_epoch + 1})"
        )

    print(f"Mean accuracy: {np.mean([r['accuracy'] for r in results]):.4f}")
    return results, models

In [ ]:
def lstm_validation_correctness(models, my_cross_validation_sets, embedding_model, batch_size=256):
    """Same as mlp_validation_correctness, for the LSTM models."""
    tokenizer = part2.clslib.BasicTokenizer(
        lowercase=True,
        remove_punct=True,
        remove_stopwords=False,
        vocab=embedding_model.vocab,
    )
    correct = np.zeros(sum(len(split["validation"]) for split in my_cross_validation_sets[0].values()), dtype=int)

    for fold_index, split in my_cross_validation_sets[0].items():
        model = models[fold_index].eval()
        model_device = next(model.parameters()).device
        loader = DataLoader(
            SentencePairSequenceDataset(split["validation"], tokenizer),
            batch_size=batch_size,
            collate_fn=lstm_collate,
        )

        predictions, labels = [], []
        with torch.no_grad():
            for sequences1, lengths1, sequences2, lengths2, batch_labels in loader:
                logprobs = model(sequences1.to(model_device), lengths1, sequences2.to(model_device), lengths2)
                predictions.append(logprobs.argmax(dim=1).cpu())
                labels.append(batch_labels)

        validation_ids = np.asarray(split["validation_ids"], dtype=int)
        correct[validation_ids] = (torch.cat(predictions) == torch.cat(labels)).numpy().astype(int)

    return correct

Glove LSTM trainable embeddings

In [9]:
glove_lstm_results, glove_lstm_models = cross_validation_train_lstm(
    embedding_model=glove,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    device=device,
    name="LSTM GloVe trainable",
)
save_fold_scores("lstm-glove100-trainable", glove_lstm_results)
save_correctness("lstm-glove100-trainable", lstm_validation_correctness(glove_lstm_models, my_cross_validation_sets, glove))
del glove_lstm_models  # each fold model holds its own copy of the 400k-word embedding table

LSTM fold 1/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 1/5: 0.7852 (best epoch 4)


LSTM fold 2/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 2/5: 0.7595 (best epoch 4)


LSTM fold 3/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 3/5: 0.7881 (best epoch 4)


LSTM fold 4/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 4/5: 0.7905 (best epoch 4)


LSTM fold 5/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 5/5: 0.7884 (best epoch 4)
Mean accuracy: 0.7823


Random LSTM trainable embeddings

In [10]:
random_lstm_results, random_lstm_models = cross_validation_train_lstm(
    embedding_model=random_embedding_model,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    device=device,
    name="LSTM random trainable",
)
save_fold_scores("lstm-random-trainable", random_lstm_results)
save_correctness("lstm-random-trainable", lstm_validation_correctness(random_lstm_models, my_cross_validation_sets, random_embedding_model))
del random_lstm_models

LSTM fold 1/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 1/5: 0.7769 (best epoch 3)


LSTM fold 2/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 2/5: 0.8184 (best epoch 4)


LSTM fold 3/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 3/5: 0.8049 (best epoch 3)


LSTM fold 4/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 4/5: 0.7704 (best epoch 3)


LSTM fold 5/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 5/5: 0.7947 (best epoch 3)
Mean accuracy: 0.7931


# LSTM with frozen embeddings
We now freeze the embeddings to see if this impacts the perforance of the models

Glve


In [11]:
glove_frozen_lstm_results, _ = cross_validation_train_lstm(
    embedding_model=glove,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    device=device,
    freeze_embeddings=True,
    name="LSTM GloVe frozen",
)
save_fold_scores("lstm-glove100-frozen", glove_frozen_lstm_results)

LSTM fold 1/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 1/5: 0.7365 (best epoch 6)


LSTM fold 2/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 2/5: 0.7348 (best epoch 6)


LSTM fold 3/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 3/5: 0.7475 (best epoch 6)


LSTM fold 4/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 4/5: 0.7478 (best epoch 7)


LSTM fold 5/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 5/5: 0.7438 (best epoch 6)
Mean accuracy: 0.7421


Random


In [12]:
random_frozen_lstm_results, _ = cross_validation_train_lstm(
    embedding_model=random_embedding_model,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    device=device,
    freeze_embeddings=True,
    name="LSTM random frozen",
)
save_fold_scores("lstm-random-frozen", random_frozen_lstm_results)

LSTM fold 1/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 1/5: 0.7198 (best epoch 15)


LSTM fold 2/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 2/5: 0.7105 (best epoch 15)


LSTM fold 3/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 3/5: 0.7118 (best epoch 13)


LSTM fold 4/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 4/5: 0.7182 (best epoch 14)


LSTM fold 5/5:   0%|          | 0/15 [00:00<?, ?it/s]

Fold 5/5: 0.7196 (best epoch 15)
Mean accuracy: 0.7160


The frozen random run had its best epoch at 13 to 15 out of 15, so it was still improving when training stopped. We rerun it with 30 epochs to give it a fair chance before comparing it with frozen GloVe.

In [ ]:
random_frozen_30_results, _ = cross_validation_train_lstm(
    embedding_model=random_embedding_model,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    device=device,
    freeze_embeddings=True,
    num_epochs=30,
    name="LSTM random frozen 30 epochs",
)
save_fold_scores("lstm-random-frozen-30ep", random_frozen_30_results)

# Bigger encoder

Does a wider LSTM help? We double `hidden_dim` from 128 to 256. This widens both the LSTM state and the hidden layer of the classifier on top. Embeddings stay trainable, as in our best setting so far.

GloVe

In [ ]:
glove_lstm_256_results, _ = cross_validation_train_lstm(
    embedding_model=glove,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    device=device,
    hidden_dim=256,
    name="LSTM GloVe trainable h256",
)
save_fold_scores("lstm-glove100-trainable-h256", glove_lstm_256_results)

Random

In [ ]:
random_lstm_256_results, _ = cross_validation_train_lstm(
    embedding_model=random_embedding_model,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    device=device,
    hidden_dim=256,
    name="LSTM random trainable h256",
)
save_fold_scores("lstm-random-trainable-h256", random_lstm_256_results)

# GloVe 300d

Does a larger embedding help? GloVe 6B comes in 50, 100, 200 and 300 dimensions, all trained on the same data with the same vocabulary. So swapping 100d for 300d changes only the embedding size. We rerun the three GloVe settings with 300d: mean-pool MLP, LSTM with frozen embeddings and LSTM with trainable embeddings.

The 300d file is about 1 GB on disk and the table takes about 0.5 GB in memory. With trainable embeddings, every fold model holds its own copy, so we do not keep the models.

In [ ]:
glove300 = load_glove(300)

Mean pooling, GloVe 300d

In [ ]:
glove300_results, _ = cross_validation_train_glove_mlp(
    embedding_model=glove300,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    num_classes=len(paws_outcome_space),
    device=device,
    name="MLP GloVe 300d",
)
save_fold_scores("mlp-glove300", glove300_results)

LSTM, GloVe 300d frozen

In [ ]:
glove300_frozen_lstm_results, _ = cross_validation_train_lstm(
    embedding_model=glove300,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    device=device,
    freeze_embeddings=True,
    name="LSTM GloVe 300d frozen",
)
save_fold_scores("lstm-glove300-frozen", glove300_frozen_lstm_results)

LSTM, GloVe 300d trainable

In [ ]:
glove300_lstm_results, _ = cross_validation_train_lstm(
    embedding_model=glove300,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    device=device,
    name="LSTM GloVe 300d trainable",
)
save_fold_scores("lstm-glove300-trainable", glove300_lstm_results)

# Training data size

With all ~39.5k training pairs per fold, random trainable embeddings did as well as GloVe. Pretrained embeddings should matter more when there is less data to learn word meanings from. We train both LSTMs (trainable embeddings) on 10%, 25% and 50% of every training fold and validate on the full validation fold.

Smaller training sets mean fewer updates per epoch, so we give them more epochs. We print a warning if the best epoch is the last one, because that means the model was still improving.

In [ ]:
import random


def subsample_training_folds(my_cross_validation_sets, fraction, seed=0):
    """Same validation folds, but only a random `fraction` of every training fold."""
    rng = random.Random(seed)
    smaller = {}
    for fold_index, split in my_cross_validation_sets[0].items():
        keep = sorted(rng.sample(range(len(split["training"])), int(fraction * len(split["training"]))))
        smaller[fold_index] = {
            **split,
            "training": [split["training"][i] for i in keep],
            "training_ids": [split["training_ids"][i] for i in keep],
        }
    return [smaller]


epochs_for_fraction = {0.1: 40, 0.25: 30, 0.5: 20}

for fraction, epochs in epochs_for_fraction.items():
    smaller_sets = subsample_training_folds(my_cross_validation_sets, fraction)
    for embedding_name, embedding_model in (("glove100", glove), ("random", random_embedding_model)):
        results, _ = cross_validation_train_lstm(
            embedding_model=embedding_model,
            my_cross_validation_sets=smaller_sets,
            num_folds=num_folds,
            device=device,
            num_epochs=epochs,
            name=f"LSTM {embedding_name} {fraction:.0%} data",
        )
        save_fold_scores(f"lstm-{embedding_name}-trainable-data{round(fraction * 100)}", results)
        if any(result["best_epoch"] == epochs - 1 for result in results):
            print(f"WARNING: {embedding_name} at {fraction:.0%} peaked at the last epoch, it may need more epochs")

In [ ]:
fold_scores = load_fold_scores()
fractions = [0.1, 0.25, 0.5, 1.0]

fig, ax = plt.subplots(figsize=(7, 4.2))
for embedding_name, label, colour in (("glove100", "GloVe 100d", TRAIN_COLOUR), ("random", "Random", VALIDATION_COLOUR)):
    names = [f"lstm-{embedding_name}-trainable-data{round(f * 100)}" for f in fractions[:-1]] + [f"lstm-{embedding_name}-trainable"]
    available = [(f, fold_scores[name]) for f, name in zip(fractions, names) if name in fold_scores]
    sizes = [f * 100 for f, _ in available]
    means = [100 * np.mean(scores) for _, scores in available]
    stds = [100 * np.std(scores, ddof=1) for _, scores in available]
    ax.errorbar(sizes, means, yerr=stds, color=colour, linewidth=2, marker="o", capsize=3, label=label)
    print(label, ", ".join(f"{s:.0f}%: {m:.1f}" for s, m in zip(sizes, means)))

ax.set_xscale("log")
ax.set_xticks([10, 25, 50, 100])
ax.set_xticklabels(["10%", "25%", "50%", "100%"])
ax.set_xlabel("share of each training fold used (100% = ~39.5k pairs)")
ax.set_ylabel("validation accuracy (%), mean +- std over 5 folds")
ax.set_title("LSTM with trainable embeddings: does GloVe help with less data?", loc="left", fontsize=11)
ax.grid(axis="y", color="#E5E7EB")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

# Variance across splits

GloVe vs random embeddings (LSTM, trainable) is our closest comparison, and one fold differed by 5.9 points. A single 5-fold split may not be enough to decide it. We repeat this comparison on two more cross-validation splits (split seeds 1 and 2, each with its own model initialisation and its own random embeddings). The Bayesian test below then uses all 3 x 5 folds (`runs=3`).

In [ ]:
split_seeds = [1, 2]  # split seed 0 is the main experiment above

for split_seed in split_seeds:
    other_split = [part1.experiment.cross_validation_split(pairs, k=num_folds, seed=split_seed)]
    for embedding_name, embedding_model in (("glove100", glove), ("random", RandomEmbeddings(glove, seed=split_seed))):
        results, _ = cross_validation_train_lstm(
            embedding_model=embedding_model,
            my_cross_validation_sets=other_split,
            num_folds=num_folds,
            device=device,
            seed=split_seed,
            name=f"LSTM {embedding_name} split {split_seed}",
        )
        save_fold_scores(f"lstm-{embedding_name}-trainable-split{split_seed}", results)

# Comparing systems: Bayesian correlated t-test with a ROPE

As in Week 1, we use the Bayesian correlated t-test (Benavoli et al., 2017) on the fold accuracies, with a **ROPE of +-0.005** (half a point of accuracy). This is about one standard error of accuracy on a validation fold of 9,880 pairs, so differences inside it are within the noise of a single fold. Same folds, same ROPE as Week 1, so we can also compare with our best Week 1 system directly.

For every comparison we report P(A better), P(practically equivalent) and P(B better). We call a difference real when one side has more than 95% of the posterior.

In [ ]:
import pandas as pd

ROPE = 0.005  # about one standard error of accuracy on a validation fold of 9,880 pairs (same as Week 1)

fold_scores = load_fold_scores()
# Best Week 1 system: the two-layer MLP on bag-of-words (Week 1 project.ipynb). Week 1 used the same folds
# (cross_validation_split on the same pairs, k=5, seed=0), so the comparison is paired fold by fold.
fold_scores["week1-two-layer-bow"] = [0.6074, 0.6076, 0.6008, 0.6095, 0.5992]

main_runs = [name for name in fold_scores if "-split" not in name and "-data" not in name]
print("Accuracy % per system (5 folds, split seed 0), mean +- std")
for name in sorted(main_runs, key=lambda name: np.mean(fold_scores[name])):
    print(f"  {name:32s} {100 * np.mean(fold_scores[name]):5.1f} +- {100 * np.std(fold_scores[name], ddof=1):.1f}")

table = pd.DataFrame(
    [[name, accuracy, 0, fold] for name in main_runs for fold, accuracy in enumerate(fold_scores[name])],
    columns=["model", "accuracy", "run", "fold"],
)
part1.analysis.plot_box_and_points(table)

comparisons = [
    # (A, B, question): B - A > 0 means B is better
    ("mlp-random", "mlp-glove100", "Does GloVe help embed-and-pool?"),
    ("week1-two-layer-bow", "mlp-glove100", "Is embed-and-pool with GloVe better than our best Week 1 system?"),
    ("mlp-glove100", "lstm-glove100-trainable", "Does a recurrent encoder beat embed-and-pool (GloVe)?"),
    ("week1-two-layer-bow", "lstm-random-trainable", "Is the LSTM better than our best Week 1 system?"),
    ("lstm-random-trainable", "lstm-glove100-trainable", "GloVe or random embeddings (LSTM, trainable)?"),
    ("lstm-random-frozen", "lstm-glove100-frozen", "GloVe or random embeddings (LSTM, frozen)?"),
    ("lstm-random-frozen-30ep", "lstm-glove100-frozen", "GloVe or random embeddings (LSTM, frozen, random given 30 epochs)?"),
    ("lstm-glove100-frozen", "lstm-glove100-trainable", "Does fine-tuning GloVe help?"),
    ("lstm-random-frozen", "lstm-random-frozen-30ep", "Does the frozen random LSTM gain from 30 epochs?"),
    ("mlp-glove100", "mlp-glove300", "300d vs 100d (embed-and-pool)"),
    ("lstm-glove100-frozen", "lstm-glove300-frozen", "300d vs 100d (LSTM, frozen)"),
    ("lstm-glove100-trainable", "lstm-glove300-trainable", "300d vs 100d (LSTM, trainable)"),
    ("lstm-glove100-trainable", "lstm-glove100-trainable-h256", "Hidden size 256 vs 128 (GloVe)"),
    ("lstm-random-trainable", "lstm-random-trainable-h256", "Hidden size 256 vs 128 (random)"),
]

print(f"\nBayesian correlated t-test, ROPE +-{ROPE}")
for name_a, name_b, question in comparisons:
    if name_a not in fold_scores or name_b not in fold_scores:
        print(f"  {question}\n    not run yet")
        continue
    if np.allclose(fold_scores[name_a], fold_scores[name_b]):
        print(f"  {question}\n    identical fold scores, so no difference to test")
        continue
    posterior = part1.analysis.BenavoliCorrelatedT(fold_scores[name_a], fold_scores[name_b], runs=1, names=[name_a, name_b])
    p_a, p_rope, p_b = posterior.probs(ROPE)
    print(f"  {question}\n    A={name_a}, B={name_b}: B-A {100 * posterior.mean_diff:+.1f} points, "
          f"P(A better) {p_a:.3f}, P(equivalent) {p_rope:.3f}, P(B better) {p_b:.3f}")

In [ ]:
# GloVe vs random (LSTM, trainable) on all three splits: 3 runs x 5 folds
glove_names = ["lstm-glove100-trainable"] + [f"lstm-glove100-trainable-split{s}" for s in (1, 2)]
random_names = ["lstm-random-trainable"] + [f"lstm-random-trainable-split{s}" for s in (1, 2)]

if all(name in fold_scores for name in glove_names + random_names):
    random_scores = [score for name in random_names for score in fold_scores[name]]
    glove_scores = [score for name in glove_names for score in fold_scores[name]]
    runs = len(glove_names)
else:
    print("Not all split seeds have been run yet, so this uses split seed 0 only.")
    random_scores, glove_scores, runs = fold_scores["lstm-random-trainable"], fold_scores["lstm-glove100-trainable"], 1

closest = part1.analysis.BenavoliCorrelatedT(random_scores, glove_scores, runs=runs, names=["Random (LSTM)", "GloVe (LSTM)"])
p_a, p_rope, p_b = closest.probs(ROPE)
print(f"A=random, B=GloVe, {runs} split(s) x {num_folds} folds: B-A {100 * closest.mean_diff:+.2f} points, "
      f"P(random better) {p_a:.3f}, P(equivalent) {p_rope:.3f}, P(GloVe better) {p_b:.3f}")
print(f"  mean accuracy: random {np.mean(random_scores):.4f}, GloVe {np.mean(glove_scores):.4f}")
fig, ax = closest.plot(ROPE)
plt.show()

# Error analysis

Which pairs does each design get wrong? We compare three systems that differ in one design choice each:

* **MLP GloVe** vs **LSTM GloVe**: mean pooling vs a recurrent encoder
* **LSTM GloVe** vs **LSTM random**: pretrained vs random embeddings

Every pair gets one out-of-fold prediction (from the fold model that did not train on it), so we can use all 49,401 pairs. Analysis features, each chosen for a reason specific to PAWS:

* **word overlap** (Jaccard of the two token sets): PAWS non-paraphrases are made by swapping words, so high overlap is where word order has to do the work
* **same bag of words**: both sentences contain exactly the same words. Mean pooling cannot separate these pairs at all
* **length** (mean number of tokens of the two sentences): longer sentences are harder to summarise in one LSTM state
* **length difference**: back-translated pairs often differ in length
* **unknown-word rate**: share of tokens without a GloVe vector (names, rare words)
* **gold label is paraphrase**: is a system biased towards one class?

In [ ]:
error_systems = ["mlp-glove100", "lstm-glove100-trainable", "lstm-random-trainable"]
correct = {name: load_correctness(name) for name in error_systems}
for name in error_systems:
    print(f"{name}: out-of-fold accuracy {correct[name].mean():.4f}")

lengths = np.array([(len(t1) + len(t2)) / 2 for t1, t2 in zip(tokens1, tokens2)])
analysis_features = {
    "word overlap": jaccard,
    "same bag of words": same_bag.astype(float),
    "length": lengths,
    "length difference": np.array([abs(len(t1) - len(t2)) for t1, t2 in zip(tokens1, tokens2)], dtype=float),
    "unknown-word rate": pair_unknown_rate,
    "gold: paraphrase": y_all.astype(float),
}

# Accuracy per gold label
print("\nAccuracy % per gold label")
for label, label_name in enumerate(["not paraphrase", "paraphrase"]):
    mask = y_all == label
    print(f"  {label_name:15s} (n={mask.sum():,}): " + ", ".join(f"{name} {100 * correct[name][mask].mean():.1f}" for name in error_systems))

# Accuracy per word-overlap bin, with identical word bags as their own group
overlap_bins = pd.cut(jaccard, [0, 0.6, 0.7, 0.8, 0.9, 1.0], include_lowest=True,
                      labels=["0.0-0.6", "0.6-0.7", "0.7-0.8", "0.8-0.9", "0.9-1.0"])
overlap_group = np.where(same_bag, "same bag of words", overlap_bins.astype(str))
by_overlap = pd.DataFrame({"overlap": overlap_group, "paraphrase": y_all, **{name: correct[name] for name in error_systems}})
summary = by_overlap.groupby("overlap").agg(pairs=("paraphrase", "size"), paraphrase_share=("paraphrase", "mean"),
                                            **{name: (name, "mean") for name in error_systems})
print("\nAccuracy per word-overlap group (Jaccard of token sets)")
print(summary.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(len(summary))
for offset, (name, colour) in zip((-0.27, 0, 0.27), zip(error_systems, ("#9CA3AF", TRAIN_COLOUR, VALIDATION_COLOUR))):
    ax.bar(x + offset, 100 * summary[name], width=0.27, color=colour, label=name)
ax.set_xticks(x)
ax.set_xticklabels([f"{group}\n(n={n:,})" for group, n in zip(summary.index, summary["pairs"])], fontsize=8)
ax.set_ylabel("accuracy (%)")
ax.set_ylim(40, 100)
ax.set_title("Accuracy by word overlap of the two sentences", loc="left", fontsize=11)
ax.grid(axis="y", color="#E5E7EB")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

**Bayesian model of correctness** (the BDA model from the tutorial, `part2.analysis.fit_bda_model`). It predicts whether a system is correct on a pair from a system-specific shift plus system-specific weights on the (standardised) features above. A weight above 0 means the system does better as that feature grows, below 0 means it struggles. Comparing weights across systems shows where the design choices differ.

MCMC on all 3 x 49k observations is slow, so we fit it on a random sample of 6,000 pairs (the same pairs for all three systems).

In [ ]:
feature_names = list(analysis_features)
feature_matrix = np.column_stack([analysis_features[name] for name in feature_names])
sample = np.random.default_rng(0).choice(len(y_all), size=min(6000, len(y_all)), replace=False)

mcmc = part2.analysis.fit_bda_model(
    num_systems=len(error_systems),
    sys_id=np.repeat(np.arange(len(error_systems)), len(sample)),
    features=np.tile(feature_matrix[sample], (len(error_systems), 1)),
    correctness=np.concatenate([correct[name][sample] for name in error_systems]),
    num_chains=2,
)

posterior_samples = mcmc.get_samples()
weights = np.asarray(posterior_samples["w"])  # [draws, systems, features]
accuracy_samples = np.asarray(posterior_samples["accuracy"])  # [draws, systems]

print("Posterior accuracy, mean [95% interval]")
for s, name in enumerate(error_systems):
    low, high = np.percentile(accuracy_samples[:, s], [2.5, 97.5])
    print(f"  {name:26s} {accuracy_samples[:, s].mean():.3f} [{low:.3f}, {high:.3f}]")

print("\nFeature weights (standardised features), mean [95% interval]")
for f, feature in enumerate(feature_names):
    cells_text = []
    for s, name in enumerate(error_systems):
        low, high = np.percentile(weights[:, s, f], [2.5, 97.5])
        cells_text.append(f"{name} {weights[:, s, f].mean():+.2f} [{low:+.2f}, {high:+.2f}]")
    print(f"  {feature:18s} " + " | ".join(cells_text))

fig, ax = plt.subplots(figsize=(8, 4))
y_positions = np.arange(len(feature_names))
for offset, (s, name), colour in zip((-0.22, 0, 0.22), enumerate(error_systems), ("#9CA3AF", TRAIN_COLOUR, VALIDATION_COLOUR)):
    low, high = np.percentile(weights[:, s, :], [2.5, 97.5], axis=0)
    mean = weights[:, s, :].mean(axis=0)
    ax.errorbar(mean, y_positions + offset, xerr=[mean - low, high - mean], fmt="o", color=colour, capsize=3, label=name)
ax.axvline(0, color="#374151", linewidth=1)
ax.set_yticks(y_positions)
ax.set_yticklabels(feature_names)
ax.invert_yaxis()
ax.set_xlabel("weight on the logit of being correct (95% interval)")
ax.set_title("Which pairs does each system get right?", loc="left", fontsize=11)
ax.grid(axis="x", color="#E5E7EB")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()